# Build DSPy modules for various tasks

In [1]:
import dspy
llama3 = dspy.LM("ollama_chat/llama3.2:1b", api_base="http://localhost:11434", api_key="")
gpt_oss = dspy.LM("ollama_chat/gpt-oss:latest", api_base="http://localhost:11434", api_key="")

## Math

In [2]:
with dspy.context(lm=llama3):
    math = dspy.ChainOfThought("question -> answer: float")
    output = math(question="Two dice are tossed. What is the probability that the sum equals two?")
    print(output)

Prediction(
    reasoning="To calculate the probability that the sum of the dice rolls equals two, we need to first determine all possible outcomes when rolling two dice. Each die has 6 faces, so there are 36 possible outcomes in total (6 x 6 = 36). Now, let's identify which of these outcomes result in a sum of two: (1, 1), (2, 0) is not valid because one side of the die shows zero; however (3, 3), (4, 2), and (5, 1) are all valid combinations. Thus, there are three favorable outcomes for our event. Therefore, the probability can be calculated as follows: probability = number of favorable outcomes / total number of possible outcomes",
    answer=0.08333333333
)


## RAG

In [4]:
def search_wikipedia(query: str) -> list[str]:
    results = dspy.ColBERTv2(url="http://20.102.90.50:2017/wiki17_abstracts")(query, k=3)
    return [x["text"] for x in results]

with dspy.context(lm=gpt_oss):
    rag = dspy.ChainOfThought("context, question -> response")
    
    question = "What's the name of the castle that David Gregory inherited?"
    output = rag(context=search_wikipedia(question), question=question)
    print(output)

KeyError: 'topk'

## Classification

In [5]:
from typing import Literal

class Classify(dspy.Signature):
    """Classify sentiment of a given sentence."""

    sentence: str = dspy.InputField()
    sentiment: Literal["positive", "negative", "neutral"] = dspy.OutputField()
    confidence: float = dspy.OutputField()

with dspy.context(lm=llama3):
    classify = dspy.Predict(Classify)
    output = classify(sentence="This book was super fun to read, though not the last chapter.")
    print(output)

Prediction(
    sentiment='neutral',
    confidence=0.5
)


## Information Extraction

In [8]:
class ExtractInfo(dspy.Signature):
    """Extract structured information from text."""

    text: str = dspy.InputField()
    title: str = dspy.OutputField()
    headings: list[str] = dspy.OutputField()
    entities: list[dict[str, str]] = dspy.OutputField(desc="a list of entities and their metadata")

with dspy.context(lm=gpt_oss):
    module = dspy.Predict(ExtractInfo)

    text = "Apple Inc. announced its latest iPhone 14 today." \
        "The CEO, Tim Cook, highlighted its new features in a press release."
    response = module(text=text)

    print(response.title)
    print(response.headings)
    print(response.entities)

Apple Inc. Announces Latest iPhone 14
['Apple Inc. Announces iPhone 14', 'CEO Tim Cook Highlights Features']
[{'name': 'Apple Inc.', 'type': 'Organization'}, {'name': 'iPhone 14', 'type': 'Product'}, {'name': 'Tim Cook', 'type': 'Person'}]


## Agents

In [10]:
def evaluate_math(expression: str):
    return dspy.PythonInterpreter({}).execute(expression)

def search_wikipedia(query: str):
    results = dspy.ColBERTv2(url="http://20.102.90.50:2017/wiki17_abstracts")(query, k=3)
    return [x["text"] for x in results]

with dspy.context(lm=gpt_oss):
    react = dspy.ReAct("question -> answer: float", tools=[evaluate_math, search_wikipedia])
    
    pred = react(question="What is 9362158 divided by the year of birth of David Gregory of Kinnairdy castle?")
    print(pred.answer)

KeyboardInterrupt: 

## Multi-Stage Pipelines

In [11]:
class Outline(dspy.Signature):
    """Outline a thorough overview of a topic."""

    topic: str = dspy.InputField()
    title: str = dspy.OutputField()
    sections: list[str] = dspy.OutputField()
    section_subheadings: dict[str, list[str]] = dspy.OutputField(desc="mapping from section headings to subheadings")

class DraftSection(dspy.Signature):
    """Draft a top-level section of an article."""

    topic: str = dspy.InputField()
    section_heading: str = dspy.InputField()
    section_subheadings: list[str] = dspy.InputField()
    content: str = dspy.OutputField(desc="markdown-formatted section")

class DraftArticle(dspy.Module):
    def __init__(self):
        self.build_outline = dspy.ChainOfThought(Outline)
        self.draft_section = dspy.ChainOfThought(DraftSection)

    def forward(self, topic):
        outline = self.build_outline(topic=topic)
        sections = []
        for heading, subheadings in outline.section_subheadings.items():
            section, subheadings = f"## {heading}", [f"### {subheading}" for subheading in subheadings]
            section = self.draft_section(topic=outline.title, section_heading=section, section_subheadings=subheadings)
            sections.append(section.content)
        return dspy.Prediction(title=outline.title, sections=sections)

with dspy.context(lm=llama3):
    draft_article = DraftArticle()
    article = draft_article(topic="World Cup 2002")
    print(article)

Prediction(
    title='FIFA World Cup 2002',
    sections=["FIFA World Cup 2002\n\nThe final was contested between Brazil and Germany. Brazil won the match 2-0.\nBrazil's 13th World Cup title marked its third consecutive championship after wins in 1958 and 1962.", '### The tournament was held in a single round-robin format, with the winner advancing directly to the semi-finals.\nBrazil was drawn against South Korea and Switzerland in the group stage.\nGermany defeated Norway 4-0 in their final group match.', '*The participating teams were:*\n  * Argentina\n  * Australia\n  * Austria\n  * Belgium\n  * Bolivia\n  * Bosnia and Herzegovina\n  * Brazil\n  * Bulgaria\n  * Cameroon\n  * Canada\n  * Czech Republic\n  * Denmark\n  * Egypt\n  * France\n  * Germany\n  * Greece\n  * Hungary\n  * Iceland\n  * Italy\n  * Japan\n  * Mexico\n  * Netherlands\n  * Norway\n  * Panama\n  * Paraguay\n  * Poland\n  * Portugal\n  * Russia\n  * Saudi Arabia\n  * Senegal\n  * Serbia and Montenegro\n  * Slovaki